# Results for model: meta/llama-3.1-405b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Feature engineering
df = df.sort(by='date_id')
top_quantile = df.groupby_rolling(index_column='date_id', period='1d', by='date_id').agg(pl.col('feature_00').quantile(0.85, interpolation='nearest')).alias('top_quantile')
df = df.join(top_quantile, on='date_id', how='left')

# Create feature columns
feature_cols = [col for col in df.columns if col.startswith('feature')]
df = df.select(feature_cols + ['top_quantile', 'responder_6'])

# Convert to pandas dataframe
df = df.to_pandas()

# Split data
X = df.drop('responder_6', axis=1)
y = df['responder_6']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost regressor
model = xgb.XGBRegressor(objective='reg:squarederror')
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')

# Train model on full data
model.fit(X, y)